## Summary of data

In [ ]:
import os
import pandas as pd
# from google.colab import drive

In [ ]:
# drive.mount('/content/drive')

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
folder_path = '../data/new'

In [ ]:
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
print(f"Found {len(csv_files)} CSV files. Starting batch processing...\n")

In [ ]:
from datetime import datetime
from IPython.display import display, Markdown
def extract_date(filename):

    date_str = filename.split('-')[0]
    try:
        return datetime.strptime(date_str, '%m%d%Y')
    except ValueError:
        return datetime.min

In [ ]:
sorted_files = sorted(csv_files, key=extract_date)

In [ ]:
threshold_date = datetime(2025, 6, 24)

In [ ]:
display(Markdown(f"## 📊 Batch Processing (Date-Specific Logic Applied)"))

# 3. Loop and Read
for file_name in sorted_files:
    file_path = os.path.join(folder_path, file_name)
    file_date = extract_date(file_name)

    # Logic: If date is on or after June 24, 2025, skip 5 rows (read from 6th line)
    if file_date and file_date > threshold_date:
        skip_rows = 5
        notice = "⚠️ **Skipping first 5 rows** (Date > 2025-06-24)"
    else:
        skip_rows = 0
        notice = "✅ Reading from the first row"

    try:
        # Using utf-8-sig for BOM and latin1 as fallback for special characters
        try:
            df = pd.read_csv(file_path, encoding='utf-8-sig', skiprows=skip_rows, on_bad_lines='skip')
        except UnicodeDecodeError:
            df = pd.read_csv(file_path, encoding='latin1', skiprows=skip_rows, on_bad_lines='skip')
            notice += " | *Loaded with Latin1 encoding*"

        display(Markdown(f"---"))
        display(Markdown(f"### 📅 File: `{file_name}`"))
        display(Markdown(notice))

        # Display metadata
        metadata = pd.DataFrame({
            "Metric": ["Total Rows", "Total Columns", "Column Names"],
            "Details": [len(df), len(df.columns), " | ".join(df.columns.tolist())]
        })
        display(metadata.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

        # Preview Data
        display(Markdown("**Data Preview (Top 2 Rows):**"))
        display(df.head(2).style.set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#4F81BD'), ('color', 'white')]}
        ]))

    except Exception as e:
        display(Markdown(f"**❌ Failed to read `{file_name}`**: {e}"))

# Clean data

In [ ]:
import os
import pandas as pd
from datetime import datetime

### clean new data

In [ ]:
import os
import pandas as pd

def clean_and_save_csv(source_dir, target_dir):
    # 1. Check and create a new directory for cleaned data
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)

    # Find all CSV files
    files = [f for f in os.listdir(source_dir) if f.endswith('.csv')]

    # Official organization name mapping dictionary
    official_map = {
        'UVA': 'University of Virginia',
        'uva': 'University of Virginia',
        'UVa': 'University of Virginia',
        'Uva': 'University of Virginia',
        'UVA SEAS': 'University of Virginia',
        'UVa - Engineering Information Technology':'University of Virginia',
        'UVA Health':'University of Virginia',
        'UVA Engineering':'University of Virginia',
        'UVA RC':'University of Virginia',
        'Univ. of Virginia':'University of Virginia',
        'VT': 'Virginia Tech',
        'vt':'Virginia Tech',
        'Virginia tech':'Virginia Tech',
        'Viginia tech':'Virginia Tech',
        'William and Mary': 'William & Mary',
        'William & Mary/Cydecor': 'William & Mary',
        'wm': 'William & Mary',
        'WM': 'William & Mary',
        'W&M (and Cydecor)': 'William & Mary',
        'W&M': 'William & Mary',
        'W&m': 'William & Mary',
        'George Mason': 'George Mason University',
        'GMU':'George Mason University',
        'gmu':'George Mason University',
        'VIMS':'William & Mary',
        'Virginia Institute of Marine Science':'William & Mary',
        'vcu':'Virginia Commonwealth University',
        'VCU':'Virginia Commonwealth University',
        'ODU':'Old Dominion University',
        'JMU':'James Madison University',
        'jmu':'James Madison University'

        #add more mappings here
    }

    all_dfs = []

    print("Starting to read files and skipping first 5 rows...")
    threshold_date = datetime(2025, 6, 24)

    # 2. Loop to read all files


    for file in files:
        file_path = os.path.join(source_dir, file)

        date_str = file.split('-')[0] # "06242025"

        try:
            file_date = datetime.strptime(date_str, '%m%d%Y')
            skip = 5 if file_date > threshold_date else 0
        except ValueError:
            skip = 0

        try:
            # Strictly skip the first skip rows, read from row skip+1
            try:
                df = pd.read_csv(file_path, skiprows=skip, encoding='utf-8-sig', on_bad_lines='skip')
            except UnicodeDecodeError:
                df = pd.read_csv(file_path, skiprows=skip, encoding='latin1', on_bad_lines='skip')

            # Create 'Name' column to match the same person across files
            if 'First Name' in df.columns and 'Last Name' in df.columns:
                df['Name'] = (df['First Name'].fillna('') + ' ' + df['Last Name'].fillna('')).str.upper().str.strip()
                df['Source_File'] =file  # Record which file the data comes from
                #print(f"Successfully read: {file}")
                print(df)
                all_dfs.append(df)
        except Exception as e:
            print(f"Read error {file}: {e}")

    if not all_dfs:
        print("No valid data found.")
        return

    print("Starting cross-file fill and generating strict 'ORG' column...")

    # 3. Merge all data to enable cross-file searching
    combined_df = pd.concat(all_dfs, ignore_index=True)

    # Extract 'Organization', clean it, and name it exactly 'ORG'
    combined_df['ORG'] = combined_df['Organization'].astype(str).str.strip().replace({'nan': None, 'None': None, '': None})

    # First replacement: convert to official names
    combined_df['ORG'] = combined_df['ORG'].replace(official_map)

    # Core logic: Group by Name, fill missing 'ORG' using records from other files
    combined_df['ORG'] = combined_df.groupby('Name')['ORG'].transform(lambda x: x.ffill().bfill())

    # Second replacement: ensure filled names are also official
    combined_df['ORG'] = combined_df['ORG'].replace(official_map)

    print("Starting column reordering and saving to new path...")

    # 4. Split combined data back to original files, reformat and save
    for file in files:
        # Extract data belonging to the current file
        file_df = combined_df[combined_df['Source_File'] == file].copy()

        if not file_df.empty:
            # Get all current column names
            cols = file_df.columns.tolist()

            # Remove temporary helper columns generated during processing
            cols.remove('Name')
            cols.remove('Source_File')

            # Temporarily take out the 'ORG' column from the list
            cols.remove('ORG')

            # Find 'Organization' index, strictly insert 'ORG' right behind it
            if 'Organization' in cols:
                idx = cols.index('Organization')
                cols.insert(idx + 1, 'ORG')
            else:
                cols.append('ORG')

            # Reorganize the dataframe with the new column order
            file_df = file_df[cols]

            # Generate new filename: original_clean.csv
            new_filename = file.replace('.csv', '.csv')
            save_path = os.path.join(target_dir, new_filename)

            # Save to the new path
            file_df.to_csv(save_path, index=False, encoding='utf-8-sig')
            print(f"Successfully saved: {new_filename}")

    print(f"\nAll cleaning done! check {target_dir}")


In [ ]:

# =Source data path
#source_folder = '../data/2025'
source_folder = '../data/new_data'
# New path for cleaned data
#target_folder = '../data/2025_Cleaned'
target_folder = '../data/new_data_cleaned'

clean_and_save_csv(source_folder, target_folder)

## fill the empty value from previous

In [ ]:
import os
import pandas as pd
from datetime import datetime

In [ ]:
import os
import pandas as pd
from datetime import datetime

def cross_validate_and_fill_org(history_dir, dir_2025, target_dir):
    # Ensure target directory exists
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)

    # --- STEP 1: Build Reference Dictionary from Previous Data (2021-2024) ---
    history_files = [f for f in os.listdir(history_dir) if f.endswith('.csv')]
    history_records = []

    for file in history_files:
        path = os.path.join(history_dir, file)
        try:
            # Read historical files
            try:
                df_hist = pd.read_csv(path, encoding='utf-8-sig', on_bad_lines='skip')
            except UnicodeDecodeError:
                df_hist = pd.read_csv(path, encoding='latin1', on_bad_lines='skip')

            # Extract standard Name and ORG to build the dictionary
            if 'First Name' in df_hist.columns and 'Last Name' in df_hist.columns:
                df_hist['Name'] = (df_hist['First Name'].fillna('') + ' ' + df_hist['Last Name'].fillna('')).str.upper().str.strip()

                # Check for ORG or Organization column
                org_col = 'ORG' if 'ORG' in df_hist.columns else 'Organization' if 'Organization' in df_hist.columns else None

                if org_col:
                    # Clean the string values to ensure blanks are treated as true Nulls
                    df_hist['Temp_ORG'] = df_hist[org_col].astype(str).str.strip().replace({'nan': None, 'None': None, '': None})
                    history_records.append(df_hist[['Name', 'Temp_ORG']])
        except Exception as e:
            print(f"Error reading history file {file}: {e}")

    # Create the lookup dictionary: {NAME: ORGANIZATION}
    if history_records:
        hist_combined = pd.concat(history_records, ignore_index=True).dropna(subset=['Temp_ORG', 'Name'])
        hist_combined = hist_combined[hist_combined['Name'] != '']
        history_dict = dict(zip(hist_combined['Name'], hist_combined['Temp_ORG']))
    else:
        history_dict = {}

    # --- STEP 2: Process 2025 Data ---
    files_2025 = [f for f in os.listdir(dir_2025) if f.endswith('.csv')]
    dfs_2025 = []
    threshold_date = datetime(2025, 6, 24)

    for file in files_2025:
        path = os.path.join(dir_2025, file)

        # Determine skip count based on filename date
        # try:
        #     date_str = file.split('-')[0]
        #     file_date = datetime.strptime(date_str, '%m%d%Y')
        #     skip = 5 if file_date >= threshold_date else 0
        # except ValueError:
        skip = 0

        try:
            # Read 2025 files dynamically based on date
            try:
                df = pd.read_csv(path, skiprows=skip, encoding='utf-8-sig', on_bad_lines='skip')
            except UnicodeDecodeError:
                df = pd.read_csv(path, skiprows=skip, encoding='latin1', on_bad_lines='skip')

            # Require First Name, Last Name, and ORG column
            #print(df)
            if 'First Name' in df.columns and 'Last Name' in df.columns and 'ORG' in df.columns:
                df['Name'] = (df['First Name'].fillna('') + ' ' + df['Last Name'].fillna('')).str.upper().str.strip()
                df['Source_File'] = file
                dfs_2025.append(df)
            else:
                print(f"Skipped {file}: Missing essential columns.")
        except Exception as e:
            print(f"Error reading 2025 file {file}: {e}")

    # --- STEP 3: Fill Missing ORG Values ---
    if not dfs_2025:
        print("No valid 2025 data processed.")
        return

    combined_2025 = pd.concat(dfs_2025, ignore_index=True)

    # Standardize empty strings to true Null values so fillna() works properly
    combined_2025['ORG'] = combined_2025['ORG'].astype(str).str.strip().replace({'nan': None, 'None': None, '': None})

    # Internal fill: If the person attended multiple 2025 events and filled ORG in one, use it for others
    combined_2025['ORG'] = combined_2025.groupby('Name')['ORG'].transform(lambda x: x.ffill().bfill())

    # External fill: Fill any remaining empty ORG slots using the historical dictionary
    combined_2025['ORG'] = combined_2025['ORG'].fillna(combined_2025['Name'].map(history_dict))

    # --- STEP 4: Save Updated Files ---
    for file in files_2025:
        file_df = combined_2025[combined_2025['Source_File'] == file].copy()

        if not file_df.empty:
            # Remove the temporary helper columns before saving
            cols_to_drop = [c for c in ['Name', 'Source_File'] if c in file_df.columns]
            file_df = file_df.drop(columns=cols_to_drop)

            # Save the file to the new directory
            new_name = file.replace('.csv', '.csv')
            save_path = os.path.join(target_dir, new_name)
            file_df.to_csv(save_path, index=False, encoding='utf-8-sig')
            print(f"Successfully processed and saved: {new_name}")

    print(f"\nAll files processed successfully! Check the directory: {target_dir}")



In [ ]:
# history_path = '../data/cleaned'
# path_2025 = '../data/2025_Cleaned'
# output_path = '../data/2025_filled_Cleaned'

# # Execute
# cross_validate_and_fill_org(history_path, path_2025, output_path)

In [ ]:
history_path = '../data/cleaned'
path_new = '../data/new_data_cleaned'
output_path = '../data/new_data_filled_cleaned'

# Execute
cross_validate_and_fill_org(history_path, path_new, output_path)

## fill_org_by_email

In [ ]:
import os
import pandas as pd

In [ ]:


def fill_org_by_email(target_dir):
    # Dictionary mapping email domains to official organization names
    domain_map = {
        'jmu.edu': 'James Madison University',
        'gmu.edu': 'George Mason University',
        'virginia.edu': 'University of Virginia',
        'uvahealth.org':'University of Virginia',
        'stanford.edu': 'Stanford University',
        'wm.edu': 'William & Mary',
        'vt.edu': 'Virginia Tech',
        'vcu.edu':'Virginia Commonwealth University'
    }

    # Find all cleaned CSV files in the target directory
    files = [f for f in os.listdir(target_dir) if f.endswith('.csv')]

    if not files:
        print("No CSV files found in the target directory.")
        return

    print("Starting email-based ORG filling process...")

    for file in files:
        file_path = os.path.join(target_dir, file)

        try:
            # Read the previously cleaned file
            try:
                df = pd.read_csv(file_path, encoding='utf-8-sig', on_bad_lines='skip')
            except UnicodeDecodeError:
                df = pd.read_csv(file_path, encoding='latin1', on_bad_lines='skip')

            # Check if both ORG and Email columns exist before proceeding
            if 'ORG' in df.columns and 'Email' in df.columns:

                # Standardize missing values in ORG to true Nulls for fillna() to work
                df['ORG'] = df['ORG'].astype(str).str.strip().replace({'nan': None, 'None': None, '': None})

                # Extract the domain from the Email column (everything after '@')
                # Convert to lowercase and strip whitespace to ensure accurate matching
                extracted_domains = df['Email'].astype(str).str.split('@').str[-1].str.lower().str.strip()

                # Map the extracted domains to the organization names
                mapped_orgs = extracted_domains.map(domain_map)

                # Fill ONLY the missing ORG values with the mapped organizations
                df['ORG'] = df['ORG'].fillna(mapped_orgs)

                # Save the updated dataframe back to the same file, overwriting it
                df.to_csv(file_path, index=False, encoding='utf-8-sig')
                print(f"Successfully processed and updated: {file}")
            else:
                print(f"Skipped {file}: Missing 'ORG' or 'Email' column.")

        except Exception as e:
            print(f"Error processing file {file}: {e}")

    print("\nEmail-based ORG filling completed successfully!")


In [ ]:

# # Define the path where your cleaned files are currently stored
# cleaned_folder_path = '../data/2025_filled_Cleaned'

# # Execute the function
# fill_org_by_email(cleaned_folder_path)

In [ ]:
# Define the path where your cleaned files are currently stored
cleaned_folder_path = '../data/new_data_filled_cleaned'

# Execute the function
fill_org_by_email(cleaned_folder_path)

In [ ]:
### manually standarize ORG such as RC- school

## search someone' s record

In [ ]:
import os
import pandas as pd
from IPython.display import display

def get_all_person_info(directory, target_first_name, target_last_name):
    # Find all CSV files in the target directory
    files = [f for f in os.listdir(directory) if f.endswith('.csv')]

    # List to store all matched records
    matched_records = []

    # Standardize the search terms (lowercase and strip spaces)
    search_first = target_first_name.strip().lower()
    search_last = target_last_name.strip().lower()

    print(f"Scanning {len(files)} files for: {target_first_name} {target_last_name}...\n")

    for file in files:
        file_path = os.path.join(directory, file)

        try:
            # Read the CSV file with fallback encoding
            try:
                df = pd.read_csv(file_path, encoding='utf-8-sig', on_bad_lines='skip')
            except UnicodeDecodeError:
                df = pd.read_csv(file_path, encoding='latin1', on_bad_lines='skip')

            # Check if necessary columns exist for matching
            if 'First Name' in df.columns and 'Last Name' in df.columns:

                # Clean the columns for accurate matching (handle NaNs, lowercase, strip spaces)
                first_name_clean = df['First Name'].fillna('').astype(str).str.strip().str.lower()
                last_name_clean = df['Last Name'].fillna('').astype(str).str.strip().str.lower()

                # Create a boolean mask to find exact matches
                match_condition = (first_name_clean == search_first) & (last_name_clean == search_last)

                # Filter the dataframe based on the condition
                person_data = df[match_condition].copy()

                # If matches are found, tag them with the source file name
                if not person_data.empty:
                    person_data['Source_File'] = file
                    matched_records.append(person_data)

        except Exception as e:
            print(f"Error processing {file}: {e}")

    # Combine and display the results
    if matched_records:
        final_result = pd.concat(matched_records, ignore_index=True)
        print(f"✅ Found {len(final_result)} record(s) across all files.")

        # Display ALL columns to show all information
        display(final_result)
        return final_result
    else:
        print("❌ No records found for this person.")
        return pd.DataFrame()

# ================= Execute =================
# Set the folder you want to search (using the cleaned data folder as default)
target_folder = '../data/all_data_filled_cleaned'

# Run the search function for Richard Dixon
richard_info = get_all_person_info(target_folder, 'Gina', 'Mancuso')

## Get all unqiue org

In [ ]:
import os
import pandas as pd

# Define the folder path
target_folder = '../data/all_data_filled_cleaned'

def get_unique_org_values(directory):
    files = [f for f in os.listdir(directory) if f.endswith('.csv')]
    unique_orgs = set()

    print(f"Analyzing {len(files)} files in {directory}...\n")

    for file in files:
        file_path = os.path.join(directory, file)
        try:
            # Speed up the process by only reading the 'ORG' column
            # Use 'utf-8-sig' to handle potential Excel BOM issues
            try:
                df = pd.read_csv(file_path, usecols=['ORG'], encoding='utf-8-sig')
            except UnicodeDecodeError:
                df = pd.read_csv(file_path, usecols=['ORG'], encoding='latin1')

            # Drop empty values and update our set of unique names
            unique_orgs.update(df['ORG'].dropna().unique())

        except ValueError:
            # Triggers if the 'ORG' column name is missing in a specific file
            print(f"⚠️ Column 'ORG' not found in: {file}")
        except Exception as e:
            print(f"❌ Error reading {file}: {e}")

    # Sort the results alphabetically for a cleaner view
    sorted_unique_orgs = sorted(list(unique_orgs))

    print(f"\n✅ Found {len(sorted_unique_orgs)} unique values in the ORG column:")
    print("-" * 50)
    for org in sorted_unique_orgs:
        print(org)
    print("-" * 50)

    return sorted_unique_orgs

# Run the search
unique_values = get_unique_org_values(target_folder)

## standarize org names

In [ ]:
import os
import pandas as pd

def unify_org_names_in_files(directory):
    # 1. Define the master mapping dictionary
    # Format: {'Incorrect or Variant Name': 'Official Target Name'}
    unification_map = {
        # George Mason
        'George Mason Uniersity': 'George Mason University',
        'George Mason university': 'George Mason University',
        'George mason university': 'George Mason University',
        'Geroge Mason University': 'George Mason University',
        'george mason university': 'George Mason University',

        # Virginia Tech
        'Virginia Tech University': 'Virginia Tech',
        'Virgnia Tech': 'Virginia Tech',
        'virginia tech': 'Virginia Tech',

        # William & Mary
        'College of William & Mary': 'William & Mary',
        'College of William and Mary': 'William & Mary',
        'W&M Physics dept': 'William & Mary',

        # University of Virginia
        'University Of Virginia': 'University of Virginia',
        # Uncomment the lines below if you want to merge UVA departments into the main university
         'University of Virginia Medical Center': 'University of Virginia',
         'University of Virginia School of Engineering': 'University of Virginia',
         'UVA Research Computing': 'University of Virginia',
         'UVA Law Library': 'University of Virginia',
         'UVA Health Culpeper Medical Center': 'University of Virginia',

        # James Madison University
        'James Madison university': 'James Madison University',

        # University of Texas
        'UNiversity of Texas at Dallas': 'University of Texas at Dallas',

        # Harvard & Stanford
        'Harvard': 'Harvard University',
        'Harvard Medical School': 'Harvard University',
        'Stanford Research Computing': 'Stanford University',

        # Organizations & Labs
        'JLab': 'Jefferson Lab',
        'Minority Serving - Cyberinfrastructure Consortium': 'Minority Serving-Cyberinfrastructure Consortium',

        # Standardizing casing for others
        'Vanderbilt university': 'Vanderbilt University',
        'linkoping university': 'Linkoping University',

        # Freelance / Self-employed
        'self': 'Independent',
        'Freelance': 'Independent'
    }

    # Find all CSV files in the target directory
    files = [f for f in os.listdir(directory) if f.endswith('.csv')]

    print(f"Starting ORG unification process for {len(files)} files...\n")

    for file in files:
        file_path = os.path.join(directory, file)

        try:
            # Read the file
            try:
                df = pd.read_csv(file_path, encoding='utf-8-sig', on_bad_lines='skip')
            except UnicodeDecodeError:
                df = pd.read_csv(file_path, encoding='latin1', on_bad_lines='skip')

            # Check if ORG column exists
            if 'ORG' in df.columns:
                # Strip leading/trailing whitespaces to avoid missed matches
                df['ORG'] = df['ORG'].astype(str).str.strip()

                # Replace string 'nan' and 'None' with actual Null values
                df['ORG'] = df['ORG'].replace({'nan': None, 'None': None, '': None})

                # Apply the unification dictionary
                # Missing keys will simply be ignored and retain their original values
                df['ORG'] = df['ORG'].replace(unification_map)

                # Save the updated dataframe back, overwriting the file
                df.to_csv(file_path, index=False, encoding='utf-8-sig')
                print(f"✅ Unified ORG names in: {file}")
            else:
                print(f"⚠️ Skipped {file}: 'ORG' column not found.")

        except Exception as e:
            print(f"❌ Error processing file {file}: {e}")

    print("\n🎉 All files have been successfully unified!")

# ================= Execute =================
# The folder where your cleaned data resides
target_folder = '../data/all_data_filled_cleaned'

# Run the unification function
unify_org_names_in_files(target_folder)

In [ ]:
import os
import pandas as pd

# Define the folder path
target_folder = '../data/all_data_filled_cleaned'

def get_unique_org_values(directory):
    files = [f for f in os.listdir(directory) if f.endswith('.csv')]
    unique_orgs = set()

    print(f"Analyzing {len(files)} files in {directory}...\n")

    for file in files:
        file_path = os.path.join(directory, file)
        try:
            # Speed up the process by only reading the 'ORG' column
            # Use 'utf-8-sig' to handle potential Excel BOM issues
            try:
                df = pd.read_csv(file_path, usecols=['ORG'], encoding='utf-8-sig')
            except UnicodeDecodeError:
                df = pd.read_csv(file_path, usecols=['ORG'], encoding='latin1')

            # Drop empty values and update our set of unique names
            unique_orgs.update(df['ORG'].dropna().unique())

        except ValueError:
            # Triggers if the 'ORG' column name is missing in a specific file
            print(f"⚠️ Column 'ORG' not found in: {file}")
        except Exception as e:
            print(f"❌ Error reading {file}: {e}")

    # Sort the results alphabetically for a cleaner view
    sorted_unique_orgs = sorted(list(unique_orgs))

    print(f"\n✅ Found {len(sorted_unique_orgs)} unique values in the ORG column:")
    print("-" * 50)
    for org in sorted_unique_orgs:
        print(org)
    print("-" * 50)

    return sorted_unique_orgs

# Run the search
unique_values = get_unique_org_values(target_folder)

## search records based on org

In [ ]:
# =====search ORG =====
import os
import pandas as pd
from IPython.display import display

def get_all_org_info(directory, target_org):
    files = [f for f in os.listdir(directory) if f.endswith('.csv')]
    matched_records = []
    search_org = target_org.strip().lower()

    print(f"Scanning {len(files)} files for ORG: {target_org}...\n")

    for file in files:
        file_path = os.path.join(directory, file)
        try:
            try:
                df = pd.read_csv(file_path, encoding='utf-8-sig', on_bad_lines='skip')
            except UnicodeDecodeError:
                df = pd.read_csv(file_path, encoding='latin1', on_bad_lines='skip')

            if 'ORG' in df.columns:
                org_clean = df['ORG'].fillna('').astype(str).str.strip().str.lower()
                match_condition = org_clean == search_org
                org_data = df[match_condition].copy()

                if not org_data.empty:
                    org_data['Source_File'] = file
                    matched_records.append(org_data)

        except Exception as e:
            print(f"Error processing {file}: {e}")

    if matched_records:
        final_result = pd.concat(matched_records, ignore_index=True)
        print(f"✅ Found {len(final_result)} record(s) across all files.")
        display(final_result)
        return final_result
    else:
        print("❌ No records found for this organization.")
        return pd.DataFrame()


In [ ]:

# ================= Execute =================
target_folder = '../data/all_data_filled_cleaned'
org_info = get_all_org_info(target_folder, 'George Mason University')

## search records based on Organization

In [ ]:
# ===== search Organization =====
import os
import pandas as pd
from IPython.display import display

def get_all_organization_info(directory, target_org):
    files = [f for f in os.listdir(directory) if f.endswith('.csv')]
    matched_records = []
    search_org = target_org.strip().lower()

    print(f"Scanning {len(files)} files for Organization: {target_org}...\n")

    for file in files:
        file_path = os.path.join(directory, file)
        try:
            try:
                df = pd.read_csv(file_path, encoding='utf-8-sig', on_bad_lines='skip')
            except UnicodeDecodeError:
                df = pd.read_csv(file_path, encoding='latin1', on_bad_lines='skip')

            if 'Organization' in df.columns:
                org_clean = df['Organization'].fillna('').astype(str).str.strip().str.lower()
                match_condition = org_clean == search_org
                org_data = df[match_condition].copy()

                if not org_data.empty:
                    org_data['Source_File'] = file
                    matched_records.append(org_data)

        except Exception as e:
            print(f"Error processing {file}: {e}")

    if matched_records:
        final_result = pd.concat(matched_records, ignore_index=True)
        print(f"✅ Found {len(final_result)} record(s) across all files.")
        display(final_result)
        return final_result
    else:
        print("❌ No records found for this organization.")
        return pd.DataFrame()


In [ ]:

# ================= Execute =================
target_folder = '../data/all_data_filled_cleaned'
org_info = get_all_organization_info(target_folder, 'USM')

## Create Gender

In [ ]:
import os
import pandas as pd

def backfill_gender_and_role_safe(source_dir, target_dir):
    # Create target directory if it doesn't exist
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
        print(f"Created new folder : {target_dir}\n")

    # Get all CSV files in the source folder
    files = [f for f in os.listdir(source_dir) if f.endswith('.csv')]
    all_dfs = []

    print(f"Step 1: Reading {len(files)} files and standardizing column names...")


    for file in files:
        file_path = os.path.join(source_dir, file)

        try:
            # Read file with multiple encoding support
            try:
                df = pd.read_csv(file_path, encoding='utf-8-sig', on_bad_lines='skip')
            except UnicodeDecodeError:
                df = pd.read_csv(file_path, encoding='latin1', on_bad_lines='skip')

            # Map long question headers to short names
            rename_dict = {}
            for col in df.columns:
                clean_col = col.strip()
                if clean_col == 'What is your gender?':
                    rename_dict[col] = 'Gender'
                elif clean_col == 'What is your role?':
                    rename_dict[col] = 'Role'

            # Apply the renaming
            if rename_dict:
                df = df.rename(columns=rename_dict)

            # Create empty columns if they don't exist (for old files)
            if 'Gender' not in df.columns:
                df['Gender'] = None
            if 'Role' not in df.columns:
                df['Role'] = None

            # Create a unique Name key for matching
            if 'First Name' in df.columns and 'Last Name' in df.columns:
                df['Name'] = (df['First Name'].fillna('') + ' ' + df['Last Name'].fillna('')).str.upper().str.strip()
                # Tag the record with its origin
                df['Source_File'] = file
                all_dfs.append(df)
            else:
                print(f"⚠️ Skipped {file}: Missing name columns. ")

        except Exception as e:
            print(f"❌ Error reading {file}: {e}")

    if not all_dfs:
        print("No valid data found to process. ")
        return

    print("\nStep 2: Cross-filling missing data using historical records...")

    # Merge all files into one big table
    combined_df = pd.concat(all_dfs, ignore_index=True)

    # Fill Gender and Role based on Name
    for col in ['Gender', 'Role']:
        # Ensure blanks are treated as true Nulls
        combined_df[col] = combined_df[col].astype(str).str.strip().replace({'nan': None, 'None': None, '': None})

        # Core logic: Group by name and propagate known values
        combined_df[col] = combined_df.groupby('Name')[col].transform(lambda x: x.ffill().bfill())

    print("\nStep 3: Saving updated data to the NEW folder...")

    # Split back to individual files and save
    for file in files:
        # Filter rows belonging to this specific file
        file_df = combined_df[combined_df['Source_File'] == file].copy()

        if not file_df.empty:
            # IMPORTANT: Define the save path using target_dir
            save_path = os.path.join(target_dir, file)

            # Drop helper columns before saving
            cols_to_drop = [c for c in ['Name', 'Source_File'] if c in file_df.columns]
            file_df = file_df.drop(columns=cols_to_drop)

            # Save the file to the new folder
            file_df.to_csv(save_path, index=False, encoding='utf-8-sig')

    print(f"\n🎉 All tasks completed! Check the new directory: {target_dir}")



In [ ]:
# Define the source and target paths
source_folder = '../data/all_data_filled_cleaned'
target_folder = '../data/all_data_cleaned_gender'

# Execute the function
backfill_gender_and_role_safe(source_folder, target_folder)

In [ ]:
import os
import pandas as pd

def backfill_and_preserve_columns(source_dir, target_dir):
    # Create target directory if it doesn't exist
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
        print(f"Created new folder  {target_dir}\n")

    files = [f for f in os.listdir(source_dir) if f.endswith('.csv')]
    all_dfs = []

    # NEW: Dictionary to memorize the exact columns each file should have

    file_structure_map = {}

    print(f"Step 1: Reading {len(files)} files and memorizing column structures...")

    for file in files:
        file_path = os.path.join(source_dir, file)

        try:
            # Read file
            try:
                df = pd.read_csv(file_path, encoding='utf-8-sig', on_bad_lines='skip')
            except UnicodeDecodeError:
                df = pd.read_csv(file_path, encoding='latin1', on_bad_lines='skip')

            # Map long question headers to short names
            rename_dict = {}
            for col in df.columns:
                clean_col = col.strip()
                if clean_col == 'What is your gender?':
                    rename_dict[col] = 'Gender'
                elif clean_col == 'What is your role?':
                    rename_dict[col] = 'Role'

            # Apply the renaming
            if rename_dict:
                df = df.rename(columns=rename_dict)

            # Create empty columns if they don't exist
            if 'Gender' not in df.columns:
                df['Gender'] = None
            if 'Role' not in df.columns:
                df['Role'] = None

            # Memorize the exact columns this specific file should output
            file_structure_map[file] = df.columns.tolist()

            # Create helper columns for matching
            if 'First Name' in df.columns and 'Last Name' in df.columns:
                df['Name'] = (df['First Name'].fillna('') + ' ' + df['Last Name'].fillna('')).str.upper().str.strip()
                df['Source_File'] = file
                all_dfs.append(df)
            else:
                print(f"Skipped {file}: Missing name columns.")

        except Exception as e:
            print(f"Error reading {file}: {e}")

    if not all_dfs:
        print("No valid data found.")
        return

    print("\nStep 2: Cross-filling missing data using historical records...")

    # Merge all files (this creates the messy extra columns temporarily)
    combined_df = pd.concat(all_dfs, ignore_index=True)

    for col in ['Gender', 'Role']:
        combined_df[col] = combined_df[col].astype(str).str.strip().replace({'nan': None, 'None': None, '': None})
        # Group by name and fill
        combined_df[col] = combined_df.groupby('Name')[col].transform(lambda x: x.ffill().bfill())

    print("\nStep 3: Filtering extra columns and saving to the NEW folder...")

    for file in files:
        # Get data belonging to this file
        file_df = combined_df[combined_df['Source_File'] == file].copy()

        if not file_df.empty:
            # Only keep the columns we memorized in Step 1!
            desired_columns = file_structure_map[file]
            file_df = file_df[desired_columns]

            # Define save path
            save_path = os.path.join(target_dir, file)

            # Save the file
            file_df.to_csv(save_path, index=False, encoding='utf-8-sig')
            print(f"Saved clean file:{file}")

    print(f"\nAll done! No messy columns were kept. Check: {target_dir}")


In [ ]:
source_folder = '../data/all_data_filled_cleaned'
target_folder = '../data/all_data_cleaned_gender_role'

backfill_and_preserve_columns(source_folder, target_folder)

## Filling Role according to the job title

In [ ]:
import os
import pandas as pd
import numpy as np

def fill_role_by_job_title(source_dir, target_dir):
    # Create target directory if it doesn't exist
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
        print(f"📁 Created new folder: {target_dir}\n")

    files = [f for f in os.listdir(source_dir) if f.endswith('.csv')]

    print(f"Step 1: Processing {len(files)} files to infer Role from Job Title...\n")

    for file in files:
        file_path = os.path.join(source_dir, file)

        try:
            # Read file with fallback encoding
            try:
                df = pd.read_csv(file_path, encoding='utf-8-sig', on_bad_lines='skip')
            except UnicodeDecodeError:
                df = pd.read_csv(file_path, encoding='latin1', on_bad_lines='skip')

            # Ensure both columns exist before attempting to process
            if 'Job Title' in df.columns and 'Role' in df.columns:

                # Standardize empty Role values to true Nulls for accurate checking
                df['Role'] = df['Role'].astype(str).str.strip().replace({'nan': None, 'None': None, '': None})

                # Create a lowercase version of Job Title for case-insensitive matching
                jt_lower = df['Job Title'].astype(str).str.lower()

                # Identify rows where the Role column is currently empty
                missing_role = df['Role'].isna()

                # --- Define Regex Matching Rules ---
                # Rule 1: Faculty
                cond_faculty = jt_lower.str.contains(r'professor', na=False, regex=True)

                # Rule 2: Staff / Postdoc / Engineer / Manager
                cond_staff = jt_lower.str.contains(r'scientist|postdoc|engineer|manager', na=False, regex=True)

                # Rule 3: Student
                # Using \b to ensure 'ms' matches exactly the degree and not words like "systems"
                # \.? handles variations like "M.S." or "Ph.D."
                cond_student = jt_lower.str.contains(r'student|ph\.?d\.?|master|\bm\.?s\.?\b', na=False, regex=True)

                # --- Apply the Rules (Only to missing Role rows) ---
                # We apply them sequentially. If a title matches multiple rules,
                # the rule applied first takes precedence.

                # Apply Faculty rule
                df.loc[missing_role & cond_faculty, 'Role'] = 'Faculty'

                # Re-calculate missing mask so we don't overwrite what we just filled
                missing_role = df['Role'].isna()

                # Apply Staff rule
                df.loc[missing_role & cond_staff, 'Role'] = 'Staff/Postdoc/Research Scientist'

                # Re-calculate missing mask
                missing_role = df['Role'].isna()

                # Apply Student rule
                df.loc[missing_role & cond_student, 'Role'] = 'Student'

                # Save the updated dataframe to the new target directory
                save_path = os.path.join(target_dir, file)
                df.to_csv(save_path, index=False, encoding='utf-8-sig')
                print(f"✅ Successfully inferred roles and saved: {file}")

            else:
                print(f"⚠️ Skipped inference for {file}: Missing 'Job Title' or 'Role' column.")
                # Save the unmodified file to the target folder anyway to keep the dataset complete
                save_path = os.path.join(target_dir, file)
                df.to_csv(save_path, index=False, encoding='utf-8-sig')

        except Exception as e:
            print(f"❌ Error processing {file}: {e}")

    print(f"\n🎉 All done! Check the updated files in: {target_dir}")



In [ ]:
# ================= Execute =================
# Set source folder to where you just saved the gender-filled files
source_folder = '../data/all_data_cleaned_gender_role'

# Set a new target folder to store these role-inferred files safely
target_folder = '../data/cleaned'

fill_role_by_job_title(source_folder, target_folder)